# 03 — Error Analysis

Post-hoc inspection of DistilBERT test errors. Run after `scripts/run_distilbert.py` has completed at least once (needs `outputs/models/distilbert/best/` + `outputs/models/distilbert/test_probs.npy`).

Sections:
1. Confusion matrix (DistilBERT vs. LogReg, side-by-side)
2. Worst-N test errors with model probability vs. Kalshi `implied_prob`
3. Seen-vs-unseen word failure mode (Mayank flagged on 2026-04-21)
4. Per-ticker breakdown

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

sys.path.insert(0, '..')
from src.constants import MODELS_DIR, SPLITS_DIR, TRANSCRIPTS_DIR
from src.data import load_transcripts

test_df = pd.read_csv(SPLITS_DIR / 'test.csv')
probs_pos = np.load(MODELS_DIR / 'distilbert' / 'test_probs.npy')
preds = (probs_pos >= 0.5).astype(int)
assert len(preds) == len(test_df), (len(preds), len(test_df))
test_df = test_df.assign(pred=preds, prob_pos=probs_pos)
test_df.head()

## 1. Confusion matrix

In [ ]:
cm = confusion_matrix(test_df['mentioned'], test_df['pred'], labels=[0, 1])
print(classification_report(test_df['mentioned'], test_df['pred'], labels=[0, 1],
                            target_names=['not_mentioned', 'mentioned'], zero_division=0))
fig, ax = plt.subplots(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt='d', cbar=False, xticklabels=['pred 0', 'pred 1'],
            yticklabels=['true 0', 'true 1'], ax=ax)
ax.set_title('DistilBERT — confusion matrix (test)')
plt.tight_layout()

## 2. Worst-N test errors

Highest-confidence wrong predictions. Compare model probability with Kalshi `implied_prob` — if both agree and both are wrong, the market genuinely mispriced the call (rare, given §4.3). If they disagree and DistilBERT is wrong, text-only signal was misleading.

In [ ]:
errors = test_df[test_df['pred'] != test_df['mentioned']].copy()
errors['confidence'] = np.where(errors['pred'] == 1, errors['prob_pos'], 1 - errors['prob_pos'])
errors = errors.sort_values('confidence', ascending=False)
errors[['ticker', 'year', 'quarter', 'word', 'mentioned', 'pred', 'prob_pos',
        'implied_prob', 'hist_rate']].head(20)

## 3. Seen-vs-unseen-word failure mode

Split errors by whether the target word ever appeared in the ticker's prior-quarter transcripts. Mayank's hypothesis: unseen words get systematically low predicted probability, which hurts No-buys when the word is about to be announced as a new product.

In [ ]:
import re

def seen_before(row):
    prior = [t['content'] for t in load_transcripts(row['ticker'])
             if (t['year'], t['quarter']) < (int(row['year']), int(row['quarter']))]
    if not prior:
        return np.nan
    pat = re.compile(r'\b' + re.escape(str(row['word'])) + r'\b', re.IGNORECASE)
    return int(any(pat.search(d) for d in prior))

test_df['word_seen_before'] = test_df.apply(seen_before, axis=1)
grouped = test_df.groupby('word_seen_before', dropna=False).agg(
    n=('mentioned', 'size'),
    accuracy=('pred', lambda s: float((s == test_df.loc[s.index, 'mentioned']).mean())),
    base_rate_mentioned=('mentioned', 'mean'),
    mean_prob_pos=('prob_pos', 'mean'),
)
grouped

## 4. Per-ticker breakdown

Which of the 15 test tickers carries most of the errors? If errors concentrate on one or two tickers, the cross-company split is exposing a specific distribution-shift failure.

In [ ]:
per_ticker = test_df.groupby('ticker').agg(
    n=('mentioned', 'size'),
    accuracy=('pred', lambda s: float((s == test_df.loc[s.index, 'mentioned']).mean())),
    n_errors=('pred', lambda s: int((s != test_df.loc[s.index, 'mentioned']).sum())),
).sort_values('n_errors', ascending=False)
per_ticker